In [56]:
# ===== Block 1: Imports =====
import os
import random
import pandas as pd
import numpy as np
from bidict import bidict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
import sklearn
import rdkit
from rdkit import Chem

from torch.utils.data import Dataset

from torch_geometric.loader import DataLoader

In [57]:
# ===== Block 2: Settings =====
CACHE_PATH = "../data/graphs_cached.pt"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("====Versions:====")
print("torch:\t", torch.__version__)
print("PyG:\t", torch_geometric.__version__)
print("scikit-learn", sklearn.__version__)
print("device:\t", DEVICE)
if torch.cuda.is_available():
    print("CUDA:\t", torch.version.cuda)

====Versions:====
torch:	 2.10.0+cu128
PyG:	 2.7.0
scikit-learn 1.8.0
device:	 cuda
CUDA:	 12.8


In [58]:
class DonorAcceptorDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df: pd.DataFrame = df.reset_index(drop=True)
        #TODO: define programatically
        self.num_features = 13
        self.num_graph_features = 250
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        donor = row["D_graph"]
        acceptor = row["A_graph"]
        
        return donor, acceptor

In [59]:
from typing import NewType
from rdkit.Chem.rdchem import BondType

AtomicNumber = NewType('AtomicNumber', int)

if os.path.exists(CACHE_PATH):
    cache = torch.load(CACHE_PATH, weights_only=False)
    df: pd.DataFrame = cache['df']
    ATOM_DICT: bidict[AtomicNumber, int] = bidict(cache['atom_vocab'])
    BOND_DICT: bidict[BondType, int] = bidict(cache['bond_vocab'])
    MAX_MOL_SIZE: int = int(cache['max_mol_size'])
else:
    raise(NotImplementedError)

In [60]:
# mols = pd.concat([df['D_mol'], df['A_mol']] )

In [61]:
n = len(df)
idx = np.arange(n)
np.random.shuffle(idx)

cut = int(0.9 * n)
train_df = df.iloc[idx[:cut]].reset_index(drop=True)
val_df   = df.iloc[idx[cut:]].reset_index(drop=True)

train_ds = DonorAcceptorDataset(train_df)
val_ds   = DonorAcceptorDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False)

print("Train/Val:", len(train_ds), len(val_ds))

Train/Val: 1597 178


In [62]:
train_ds[0][0]

Data(x=[82, 7], edge_index=[2, 182], edge_attr=[182, 7], graph_attr=[1, 250])

# VGAE
[Bresson and Laurent, 2019](https://ui.adsabs.harvard.edu/link_gateway/2019arXiv190603412B/doi:10.48550/arXiv.1906.03412)


In [63]:
# ==== Encoder ===== 
from torch_geometric.nn import GATConv

# May redo this to use ConvNet as per the paper
class MolecularEncoder(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, graph_in_dim, latent_dim, hidden_dim):
        super().__init__()

        # Broadcast graph features
        in_dim = node_in_dim + graph_in_dim

        self.conv1 = GATConv(in_dim, hidden_dim, edge_dim=edge_in_dim)

        self.conv_mu = GATConv(hidden_dim, latent_dim, edge_dim=edge_in_dim)
        self.conv_logstd = GATConv(hidden_dim, latent_dim, edge_dim=edge_in_dim)

    def forward(self, x, edge_index, edge_attr, graph_attr, batch):
        
        # Broadcast graph attributes
        graph_attr_batch = graph_attr[batch]
        x = torch.cat([x, graph_attr_batch], dim=-1)

        x = F.relu(self.conv1(x, edge_index, edge_attr))

        mu = self.conv_mu(x, edge_index, edge_attr)
        logstd = self.conv_logstd(x, edge_index, edge_attr)

        return mu, logstd

In [ ]:
# ==== Decoder =====
from bidict._bidict import bidict
from torch import Tensor
from torch_geometric.nn import MLP, GATv2Conv


class MolecularDecoder(nn.Module):
    def __init__(self, 
                latent_dim: int, 
                emb_dim: int, 
                gat_hidden_dim=32, 
                max_mol_size: int = MAX_MOL_SIZE,
                atom_dict=ATOM_DICT, 
                edge_dict=BOND_DICT):
        super().__init__()
        self.atom_dict: bidict[AtomicNumber, int] = atom_dict
        self.edge_dict: bidict[BondType, int] = edge_dict
        self.max_size: int = max_mol_size
        self.emb_dim: int = emb_dim

        # BoA MLP: z -> soft count for each (atom_type, count_bucket)
        self.boa_mlp: MLP = MLP(in_channels=latent_dim,
                        out_channels=self.atom_dict_size * self.max_size,
                        hidden_channels=64,
                        num_layers=1)

        # Separate atom-type and positional embeddings (Breaking the Symmetry,
        # Bresson & Laurent 2019): h_i = E_atom[type(i)] + E_pos[position(i)]
        self.atom_emb: nn.Embedding = nn.Embedding(
            num_embeddings=self.atom_dict_size,
            embedding_dim=self.emb_dim)
        self.pos_emb: nn.Embedding = nn.Embedding(
            num_embeddings=self.max_size,
            embedding_dim=self.emb_dim)

        # Edge projection: every edge receives the same Uz (per paper)
        self.edge_emb: nn.Linear = nn.Linear(in_features=latent_dim,
                                            out_features=latent_dim,
                                            bias=False)

        # TODO: change to take variable number of layers
        self.conv1: GATv2Conv = GATv2Conv(in_channels=emb_dim,
                                        out_channels=gat_hidden_dim,
                                        edge_dim=latent_dim)
        self.conv2: GATv2Conv = GATv2Conv(in_channels=gat_hidden_dim,
                                        out_channels=emb_dim,
                                        edge_dim=latent_dim)

        # Per-edge bond type classifier: scores over edge_dict_size classes
        self.edge_mlp: MLP = MLP(in_channels=emb_dim,
                                out_channels=self.edge_dict_size,
                                hidden_channels=64,
                                num_layers=1)

        
    def forward(self, z: Tensor):
        """
        z: [latent_dim] — single graph-level latent vector.
        Returns node embeddings and per-edge bond type scores.
        """
        # ── Atom Generation (Bag of Atoms) ──────────────────────────────────
        # Reshape to [atom_dict_size, max_size]: each row is a soft count distribution
        # over how many of that atom type to include.
        z_soft_boa: Tensor = self.boa_mlp(z).reshape([self.atom_dict_size, self.max_size])
        z_boa = z_soft_boa.max(dim=1).indices  # [atom_dict_size] — count per atom type
        del z_soft_boa

        num_atoms = int(torch.sum(z_boa))
        atom_list = torch.arange(self.atom_dict_size, device=z.device)
        # atom_list: atom type index for each atom in the generated molecule
        # may want to look for an in-place solution
        atom_list = torch.repeat_interleave(atom_list, z_boa).long()  # [num_atoms]

        # ── Breaking the Symmetry ────────────────────────────────────────────
        # Atoms of the same type would otherwise be identical. Give each a
        # positional index cycling within its type group (0, 1, ..., count-1).
        max_duplicate_atoms = int(torch.max(z_boa))
        pos_indices = torch.arange(num_atoms, device=z.device) % max_duplicate_atoms

        # h_i = E_atom[type(i)] + E_pos[position(i)]
        x = self.atom_emb(atom_list) + self.pos_emb(pos_indices)  # [num_atoms, emb_dim]

        # ── Bond Generation: fully-connected graph ───────────────────────────
        # Build sparse [2, num_atoms²] edge_index (PyG format)
        row = torch.arange(num_atoms, device=z.device).repeat_interleave(num_atoms)
        col = torch.arange(num_atoms, device=z.device).repeat(num_atoms)
        edge_index = torch.stack([row, col], dim=0)  # [2, num_atoms²]

        # Every edge gets the same projected latent: Uz  (per paper)
        edge_feat = self.edge_emb(z)                                       # [latent_dim]
        edge_attr = edge_feat.unsqueeze(0).expand(num_atoms * num_atoms, -1)  # [num_atoms², latent_dim]

        x = self.conv1(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_attr)

        # Bond type prediction per edge: sum node features at each endpoint
        edge_scores = self.edge_mlp(x[row] + x[col])  # [num_atoms², edge_dict_size]

        return x, edge_scores, edge_index


    @property
    def atom_dict_size(self) -> int:
        return len(self.atom_dict)
    
    @property
    def edge_dict_size(self) -> int:
        return len(self.edge_dict)
    
    @property
    def max_mol_size(self) -> int:
        return self.max_size

    @property
    def max_edges(self) -> int:
        s = self.max_size
        return s * s


In [65]:
from torch_geometric.nn import global_mean_pool

class MolecularVGAE(nn.Module):
    def __init__(self,
                encoder: MolecularEncoder,
                decoder: MolecularDecoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def reparameterize(self, mu: Tensor, logstd: Tensor) -> Tensor:
        if self.training:
            std = torch.exp(logstd)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def encode(self, data):
        """Returns graph-level mu and logstd pooled from per-node encoder output."""
        mu, logstd = self.encoder(
            data.x, data.edge_index, data.edge_attr, data.graph_attr, data.batch
        )
        # Pool per-node latents -> graph-level vectors [batch_size, latent_dim]
        mu_graph     = global_mean_pool(mu,     data.batch)
        logstd_graph = global_mean_pool(logstd, data.batch)
        return mu_graph, logstd_graph

    def forward(self, data):
        """
        Returns z [batch_size, latent_dim], mu, logstd.
        Decode each z separately (decoder operates per molecule).
        """
        mu, logstd = self.encode(data)
        z = self.reparameterize(mu, logstd)
        return z, mu, logstd

    def kl_loss(self, mu: Tensor, logstd: Tensor) -> Tensor:
        return -0.5 * torch.mean(
            torch.sum(1 + 2 * logstd - mu ** 2 - logstd.exp() ** 2, dim=1)
        )


In [67]:
torch.cuda.memory._record_memory_history()

mol0 = train_ds[0][0]

latent_dim = 32

encoder = MolecularEncoder(mol0.x.size(1), mol0.edge_attr.size(1), graph_in_dim=mol0.graph_attr.size(1), latent_dim=latent_dim, hidden_dim=64)
decoder = MolecularDecoder(latent_dim=latent_dim, emb_dim=32)

model = MolecularVGAE(encoder, decoder)

model = model.to(DEVICE)

model.train()

try:
    for batch in train_loader:
        # TODO: rewrite data loaders to be all molecules rather than donor/acceptor
        donor_batch, _ = batch

        donor_batch = donor_batch.to(DEVICE)
        # acceptor_batch = acceptor_batch.to(DEVICE)

        z, mu, logstd = model(donor_batch)

        x, edge_scores, edge_index = model.decoder(z[0])
        print(x, edge_scores, edge_index)

except:
    torch.cuda.memory._dump_snapshot("my_snapshot.pickle")


    


